# Heart Disease Classification — Modelling

This notebook implements and evaluates machine learning models for binary heart disease prediction.


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

In [3]:
heart_disease = fetch_ucirepo(id=45)
df = heart_disease.data.features.copy()
df['target'] = heart_disease.data.targets
df = df.fillna(df.median(numeric_only=True))
print(df.head())
print(df.info())

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  target  
0  0.0   6.0       0  
1  3.0   3.0       2  
2  2.0   7.0       1  
3  0.0   3.0       0  
4  0.0   3.0       0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64

In [4]:
df['target_binary'] = df['target'].apply(lambda x: 0 if x == 0 else 1)
print(df['target_binary'].value_counts())

target_binary
0    164
1    139
Name: count, dtype: int64


## Target Transformation

The original dataset contains 5 classes (0–4), representing the severity of heart disease.

For this project, we converted it into a binary classification problem:
- 0 means no heart disease
- 1 means presence of heart disease (values 1–4)

This simplifies the modelling process and aligns with the project objective.

## Feature Selection and Train-Test Split

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['target', 'target_binary'])
y = df['target_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (242, 13)
X_test shape: (61, 13)
y_train shape: (242,)
y_test shape: (61,)


## Feature Scaling

In this section, we standardize the features using StandardScaler.
Scaling is important for models like KNN and SVM, as they are sensitive to the magnitude of feature values.

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train_scaled[:5])

[[-2.83850353  0.72250438 -1.10468954 -0.12598175 -0.86414249 -0.39735971
   1.01249171  2.31447034 -0.71589105 -0.87357299 -0.96343165 -0.71586852
  -0.86916075]
 [ 0.24135234  0.72250438 -0.09205746  0.97465301 -2.48363748  2.51661148
  -0.99589349  1.02124161 -0.71589105 -0.70485418 -0.96343165  0.40181007
   1.20824075]
 [ 1.56129057  0.72250438 -2.11732162  1.52497038 -0.2412598   2.51661148
   1.01249171 -0.85171034 -0.71589105 -0.78921359  0.65566876  0.40181007
  -0.86916075]
 [ 1.12131116 -1.38407465 -0.09205746  1.52497038  2.3748475  -0.39735971
   1.01249171  0.04017154 -0.71589105 -0.19869777 -0.96343165 -0.71586852
  -0.86916075]
 [-0.30862192  0.72250438  0.92057462 -1.33667998 -0.26202255  2.51661148
  -0.99589349 -0.13820484 -0.71589105 -0.78921359 -0.96343165  2.63716726
   1.20824075]]


## Baseline Model: Logistic Regression

In this section, we implement Logistic Regression as the baseline model.
This model serves as a simple linear benchmark against which more complex models will be compared.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print("Logistic Regression Results:")
print("Accuracy:", accuracy_lr)
print("Precision:", precision_lr)
print("Recall:", recall_lr)
print("F1-score:", f1_lr)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Results:
Accuracy: 0.8852459016393442
Precision: 0.8787878787878788
Recall: 0.90625
F1-score: 0.8923076923076924

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.86      0.88        29
           1       0.88      0.91      0.89        32

    accuracy                           0.89        61
   macro avg       0.89      0.88      0.88        61
weighted avg       0.89      0.89      0.89        61



## Baseline Model Results

The Logistic Regression baseline model showed strong performance on the test set.

- Accuracy: 0.885
- Precision: 0.879
- Recall: 0.906
- F1-score: 0.892

The recall score is especially important in this medical classification task, as it indicates that the model successfully identified most patients with heart disease. Overall, Logistic Regression provides a strong and interpretable baseline for comparison with more complex models.

## Additional Model 1: K-Nearest Neighbors

In this section, we implement the K-Nearest Neighbors (KNN) algorithm.
KNN is a distance-based model that classifies a data point based on the majority class of its nearest neighbors.

In [8]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

print("KNN Results:")
print("Accuracy:", accuracy_knn)
print("Precision:", precision_knn)
print("Recall:", recall_knn)
print("F1-score:", f1_knn)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_knn))

KNN Results:
Accuracy: 0.9180327868852459
Precision: 0.9354838709677419
Recall: 0.90625
F1-score: 0.9206349206349206

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.93      0.92        29
           1       0.94      0.91      0.92        32

    accuracy                           0.92        61
   macro avg       0.92      0.92      0.92        61
weighted avg       0.92      0.92      0.92        61



## KNN Model Results

- Accuracy: 0.918
- Precision: 0.935
- Recall: 0.906
- F1-score: 0.920

The model achieved higher accuracy and precision, indicating better overall classification performance. In particular, the higher precision suggests that KNN makes fewer false positive predictions.

Since KNN is a non-linear model, it is able to capture more complex patterns in the data compared to Logistic Regression. This suggests that the relationship between the features and heart disease is not purely linear.

Overall, KNN provides a strong improvement over the baseline model.

## Additional Model 2: Support Vector Machine (SVM)

In [9]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf')
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("SVM Results:")
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

SVM Results:
Accuracy: 0.9016393442622951
Precision: 0.9333333333333333
Recall: 0.875
F1-score: 0.9032258064516129

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.93      0.90        29
           1       0.93      0.88      0.90        32

    accuracy                           0.90        61
   macro avg       0.90      0.90      0.90        61
weighted avg       0.90      0.90      0.90        61



## SVM Model Results

- Accuracy: 0.902
- Precision: 0.933
- Recall: 0.875
- F1-score: 0.903

The model achieved high precision, which means that when it predicts heart disease, it is often correct. This is valuable because it reduces the number of false positive cases.

However, its recall is slightly lower than that of Logistic Regression and KNN, which means it missed more actual positive cases. In a medical context, this can be an important limitation because failing to identify a patient with heart disease may have serious consequences.

Overall, SVM performed well and provided competitive results, but its lower recall makes it slightly less suitable than KNN for this specific task.